In [1]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/training_set.parquet")

print("...Info...")
print(df.info())
print("\n...Tail...")
print(df[["date", "target"]].tail())

print("\n...Samples...")
print(df.sample(n=3))


print("\n...Target vars...")
print(df[["date", "target"]].iloc[0])

print("\n...Looking at points...")
print(df["day"].iloc[0][0])
print(df["day"].iloc[1][0])
print(df["day"].iloc[2][0])

print("...Looking at news...")
print(df["news"].sample(n=5))




...Info...
<class 'pandas.DataFrame'>
RangeIndex: 304 entries, 0 to 303
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   date    304 non-null    str   
 1   hour    304 non-null    object
 2   day     304 non-null    object
 3   week    304 non-null    object
 4   month   304 non-null    object
 5   news    304 non-null    object
 6   target  304 non-null    int64 
 7   params  304 non-null    object
dtypes: int64(1), object(6), str(1)
memory usage: 22.1+ KB
None

...Tail...
           date  target
299  2025-12-22       1
300  2025-12-26       0
301  2025-12-29       0
302  2025-12-30       0
303  2025-12-31       1

...Samples...
           date                                               hour  \
20   2024-10-29  [{'open': 0.0008510638811566757, 'high': 0.004...   
264  2025-10-29  [{'open': 0.005923395555626097, 'high': 0.0095...   
115  2025-03-25  [{'open': -0.0003272327822950048, 'high': 0.00...   

                    

In [7]:
df = pd.read_parquet("../data/processed/training_set_with_semantics.parquet")

print("...Info...")
print(df.info())
print("\n...Head...")

print("\n...Samples...")

print("...Looking at news...")
print(df["news"].iloc[0])
print(df["news"].iloc[1])


...Info...
<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   date    244 non-null    str   
 1   hour    244 non-null    object
 2   day     244 non-null    object
 3   week    244 non-null    object
 4   month   244 non-null    object
 5   news    244 non-null    object
 6   target  244 non-null    int64 
dtypes: int64(1), object(5), str(1)
memory usage: 15.9+ KB
None

...Head...

...Samples...
...Looking at news...
[{'timestamp': datetime.datetime(2024, 10, 26, 0, 0, tzinfo=<DstTzInfo 'US/Eastern' EDT-1 day, 20:00:00 DST>), 'count': 5, 'samples': 5, 'semantics': array([ 3.03154774e-02, -5.18196151e-02,  2.22534332e-02,  3.92740965e-03,
         7.19595402e-02, -4.32658195e-02,  7.49585629e-02, -6.97394833e-03,
         4.96524163e-02, -4.06313911e-02,  3.23166624e-02,  4.67526913e-02,
         3.10439505e-02, -3.54374349e-02,  5.29174227e-03, -1.85943898e-02,
      

In [2]:
print("looking at hours...")
print(df["hour"].describe())
print(df["hour"].iloc[0])
print(df["hour"].iloc[100])

looking at hours...
count                                                   304
unique                                                  304
top       [{'open': -0.0011422045787769408, 'high': 0.00...
freq                                                      1
Name: hour, dtype: object
[{'open': -0.0011422045787769408, 'high': 0.0028060348120730006, 'low': -0.008182312702612844, 'close': 0.0014916863205602672, 'volume': 12.460067367501491, 'vwap': -0.001197590727945949, 'transactions': 8.055475141757274, 'raw_close': 228.1}
 {'open': -8.768468592576189e-05, 'high': 0.005465091752838648, 'low': -0.004085493415213079, 'close': -0.0030295735670044137, 'volume': 15.251529646413372, 'vwap': -9.163067757645237e-05, 'transactions': 11.036436853608688, 'raw_close': 227.41}
 {'open': 8.794301298438365e-05, 'high': 8.794301298438365e-05, 'low': -0.007414960927851524, 'close': -0.0009898921825927103, 'volume': 15.526221202441596, 'vwap': -0.0039628127782959125, 'transactions': 11.802979721567064, 

In [ ]:
sorting_errors = 0
missing_news_buckets = 0
missing_news_days = 0
missing_news_weekend = 0

for idx, row in df.iterrows():
    # Extract just the timestamps from the 8 buckets
    timestamps = [bucket['timestamp'] for bucket in row['news']]
    # Check if the list is strictly ascending
    is_sorted = all(timestamps[i] <= timestamps[i+1] for i in range(len(timestamps)-1))
        
    if not is_sorted:
        sorting_errors += 1
    
    for bucket in row["news"]:
        if bucket["count"] == 0:
            if bucket['timestamp'].weekday() < 5:
                missing_news += 1
            else:
                missing_news_weekend += 1

if sorting_errors == 0:
    print("SUCCESS: 100 percent of rows are perfectly chronological.")
else:
    print(f"ERROR: Found {sorting_errors} rows out of order.")

print(f"Missing news: {missing_news}")
print(f"Missing news weekend: {missing_news_weekend}")

SUCCESS: 100 percent of rows are perfectly chronological.
Missing news: 941
Missing news weekend: 19


In [ ]:
# AI generated validaton

print("--- 1. Target Class Balance ---")
# A healthy, upward-trending stock like AAPL should be roughly 52% to 55% Up days (1s)
# If this is 100% or 0%, the logic is fundamentally broken.
balance = df['target'].value_counts(normalize=True).mul(100).round(2)
print(balance.astype(str) + '%')

print("\n--- 2. Logical Verification (The Eye Test) ---")
# Your pipeline appended today's 8:00 AM to 2:59 PM bars to the end of the 'hour' array.
# Therefore, the very last dictionary in the 'hour' list (index -1) is the 2:00 PM bar!
df['today_2pm_close'] = df['hour'].apply(lambda x: x[-1]['raw_close'])

# Shift the column up by 1 to perfectly align "Tomorrow's 2PM Close" on the same row
df['tomorrow_2pm_close'] = df['today_2pm_close'].shift(-1)

# Calculate what the target *should* be based on absolute math
df['calculated_target'] = (df['tomorrow_2pm_close'] > df['today_2pm_close']).astype(int)

# Print 5 random rows to visually verify the numbers
check_cols = ['date', 'today_2pm_close', 'tomorrow_2pm_close', 'target', 'calculated_target']
print(df[check_cols].sample(n=5))

print("\n--- 3. Mismatch Error Rate ---")
# We exclude the very last row (iloc[:-1]) because 'tomorrow' doesn't exist yet
check_df = df.iloc[:-1]
errors = (check_df['target'] != check_df['calculated_target']).sum()

if errors == 0:
    print(f"Mismatch Count: {errors} ✅ FLawless pipeline!")
else:
    print(f"Mismatch Count: {errors} ❌ Uh oh, we have a leak.")

print("\n--- 4. The Edge Case (Very Last Row) ---")
# Verify your dummy '1' applied correctly to the final day
print(df[['date', 'today_2pm_close', 'target']].tail(1))

--- 1. Target Class Balance ---
target
1    54.92%
0    45.08%
Name: proportion, dtype: str

--- 2. Logical Verification (The Eye Test) ---
           date  today_2pm_close  tomorrow_2pm_close  target  \
195  2025-08-25          227.550             227.970       1   
86   2025-03-17          214.550             212.215       0   
198  2025-08-28          232.870             232.150       0   
202  2025-09-04          238.475             239.955       1   
93   2025-03-26          221.575             224.106       1   

     calculated_target  
195                  1  
86                   0  
198                  0  
202                  1  
93                   1  

--- 3. Mismatch Error Rate ---
Mismatch Count: 0 ✅ FLawless pipeline!

--- 4. The Edge Case (Very Last Row) ---
           date  today_2pm_close  target
243  2025-10-31           271.67       1
